# Test: Cosine No-Freeze Model

加载 `best_vit_model_cosine_no-freeze.pth`，在测试集上评估 Test Loss 和 Accuracy。

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ── 配置 ──────────────────────────────────────────
MODEL_PATH = 'best_vit_model_cosine_no-freeze.pth'
DATA_DIR   = '.'           # 包含 test/ 文件夹的目录
BATCH_SIZE = 64
MODEL_NAME = 'google/vit-base-patch16-224'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Model file exists: {os.path.exists(MODEL_PATH)}')

In [ ]:
class WildfireDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        for class_name, label in {'wildfire': 1, 'nowildfire': 0}.items():
            class_path = os.path.join(data_dir, class_name)
            if os.path.exists(class_path):
                for img_name in os.listdir(class_path):
                    if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.image_paths.append(os.path.join(class_path, img_name))
                        self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), color='black')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]


# 准备测试数据
processor  = ViTImageProcessor.from_pretrained(MODEL_NAME)
size       = processor.size['height']
transform  = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

test_dir     = os.path.join(DATA_DIR, 'test')
test_dataset = WildfireDataset(test_dir, transform=transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Test set size: {len(test_dataset)}')

In [ ]:
# 加载 checkpoint
checkpoint     = torch.load(MODEL_PATH, map_location=device)
freeze_strategy = checkpoint.get('freeze_strategy', 'no_freeze')
saved_model    = checkpoint.get('model_name', MODEL_NAME)

print(f'Freeze strategy : {freeze_strategy}')
print(f'Best epoch      : {checkpoint.get("epoch")}')
print(f'Best val loss   : {checkpoint.get("best_val_loss", "N/A"):.4f}')
print(f'Best val acc    : {checkpoint.get("best_val_acc", "N/A"):.4f}')

# 建立模型并载入权重
model = ViTForImageClassification.from_pretrained(
    saved_model,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: 'nowildfire', 1: 'wildfire'},
    label2id={'nowildfire': 0, 'wildfire': 1},
)
if freeze_strategy == 'freeze_8_layers':
    for i, layer in enumerate(model.vit.encoder.layer):
        if i < 8:
            for p in layer.parameters():
                p.requires_grad = False

model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()
print('Model loaded successfully.')

In [ ]:
criterion  = nn.CrossEntropyLoss()
total_loss = 0.0
all_labels = []
all_preds  = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images, labels = images.to(device), labels.to(device)
        outputs = model(pixel_values=images)
        loss    = criterion(outputs.logits, labels)
        total_loss += loss.item() * images.size(0)
        preds   = outputs.logits.argmax(dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_loss = total_loss / len(test_loader.dataset)
test_acc  = accuracy_score(all_labels, all_preds)
test_recall = recall_score(all_labels, all_preds, average='binary')
conf_mat  = confusion_matrix(all_labels, all_preds)

print(f'\n{"="*40}')
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Recall   : {test_recall:.4f}')
print(f'Confusion Matrix:\n{conf_mat}')
print(f'{"="*40}')